In [1]:
# Task 4.1

import sqlite3

conn = sqlite3.connect("villaBookings.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE Villa(
    villaID INTEGER PRIMARY KEY,
    villaName TEXT NOT NULL,
    country TEXT NOT NULL,
    cost INTEGER NOT NULL
)
""")

cur.execute("""
CREATE Table CustomerBooking(
    bookingID INTEGER PRIMARY KEY,
    customerID INTEGER NOT NULL,
    villaID INTEGER NOT NULL REFERENCES Villa(villaID),
    startDate TEXT NOT NULL,
    numberOfDays INTEGER NOT NULL
)
""")

conn.commit()
conn.close()

In [2]:
# Task 4.2

villa_file = open("villas.txt", 'r')
customerBooking_file = open("customerBookings.txt", 'r')

villa_data = []
for line in villa_file:
    villa_data.append(line.strip().split(','))

customerBooking_data = []
for line in customerBooking_file:
    customerBooking_data.append(line.strip().split(','))

villa_file.close()
customerBooking_file.close()

conn = sqlite3.connect("villaBookings.db")
cur = conn.cursor()

cur.executemany("""
INSERT INTO Villa(villaID, villaName, country, cost) VALUES (?, ?, ?, ?)
""", villa_data)

cur.executemany("""
INSERT INTO CustomerBooking(bookingID, customerID, villaID, startDate, numberOfDays) VALUES (?, ?, ?, ?, ?)
""", customerBooking_data)

conn.commit()
conn.close()

In [9]:
# Task 4.3

conn = sqlite3.connect("villaBookings.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS Villa_Booking(
    villaID INTEGER NOT NULL REFERENCES Villa(villaID),
    bookedDate TEXT NOT NULL
)
""")

conn.commit()

cur.execute("""
SELECT villaID, startDate, numberOfDays FROM CustomerBooking
""")

rows = cur.fetchall()
# print(rows)

villaBookings = []

for villaID, startDate, numberOfDays in rows:
    for i in range(numberOfDays):
        date = str(int(startDate[:2]) + i).zfill(2) + startDate[2:]
        villaBookings.append([villaID, date])

# print(villaBookings)

cur.executemany("""
INSERT INTO Villa_Booking(villaID, bookedDate) VALUES (?, ?)
""", villaBookings)

conn.commit()
conn.close()

In [17]:
# Task 4.4

villaName = input("Enter the name of the villa you would like to book: ")
startDateMonth = input("Enter the start date of your booking (month): ")
startDateDay = int(input("Enter the start date of your booking (day): "))
numberOfDays = int(input("Enter the number of days you would like to book: "))

user_dates = []

for i in range(numberOfDays):
    user_dates.append(f"{startDateDay + i}-{startDateMonth}")

# print(user_dates)

conn = sqlite3.connect("villaBookings.db")
cur = conn.cursor()

cur.execute("""
SELECT bookedDate FROM Villa_Booking WHERE Villa_Booking.villaID = (
    SELECT villaID FROM Villa WHERE Villa.villaName = ?
)
""", (villaName, ))

rows = cur.fetchall()
booked_dates = []

for date in rows:
    booked_dates.append(date[0])

# print(booked_dates)
# print(len(booked_dates))

avaliable_dates = []
unavaliable_dates = []

for date in user_dates:
    if date not in booked_dates:
        avaliable_dates.append(date)
    else:
        unavaliable_dates.append(date)

print("\nAvaliable dates:")
for date in avaliable_dates:
    print(date)

print("\nUnavaliable dates:")
for date in unavaliable_dates:
    print(date)

conn.close()

Enter the name of the villa you would like to book:  Dolphin
Enter the start date of your booking (month):  Apr
Enter the start date of your booking (day):  8
Enter the number of days you would like to book:  4



Avaliable dates:
8-Apr
9-Apr

Unavaliable dates:
10-Apr
11-Apr
